In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # <-- important change

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
import json

# ----------------------------
# 1. Config
# ----------------------------
CSV_PATH = "C:\\Users\\mamat\\IEEE24\\nvd_2025_cvss3.csv"  # path to your CSV
TEXT_COL = "description"
LABEL_COL = "S"                 # here we train AC; change for AV, PR, etc.
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 4
BATCH_SIZE = 4
EPOCHS = 2                       # you can change this
LR = 2e-5
OUTPUT_DIR = f"bert_{LABEL_COL.lower()}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------
# 2. Load data
# ----------------------------
df = pd.read_csv(CSV_PATH)

# Drop rows with missing text or label
df = df.dropna(subset=[TEXT_COL, LABEL_COL])

texts = df[TEXT_COL].astype(str).tolist()
labels_raw = df[LABEL_COL].astype(str).tolist()

# ----------------------------
# 3. Encode labels (e.g., LOW/HIGH → 0/1)
# ----------------------------
unique_labels = sorted(list(set(labels_raw)))
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}

labels = [label2id[lbl] for lbl in labels_raw]

print("Label mapping:", label2id)

# Save mapping for later inference
with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)

# ----------------------------
# 4. Train/validation split
# ----------------------------
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

# ----------------------------
# 5. Dataset & DataLoader
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CVSSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = CVSSDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = CVSSDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ----------------------------
# 6. Model
# ----------------------------
num_labels = len(unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

# ----------------------------
# 7. Optimizer & scheduler
# ----------------------------
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

# ----------------------------
# 8. Training + validation
# ----------------------------
def train_epoch(loader):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)

def eval_epoch(loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)

    avg_loss = total_loss / len(loader)
    return avg_loss, all_preds, all_labels

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss, val_preds, val_true = eval_epoch(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Val loss:   {val_loss:.4f}")
    print(classification_report(val_true, val_preds, target_names=unique_labels))

# ----------------------------
# 9. Save model
# ----------------------------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model for {LABEL_COL} saved in {OUTPUT_DIR}")
